In [1]:
import numpy as np
import pandas as pd
import featuretools as ft
from woodwork.table_accessor import WoodworkTableAccessor
from woodwork.column_accessor import WoodworkColumnAccessor

# Патч для DataFrame.ww
_orig_tab_init = WoodworkTableAccessor.__init__
def _patched_tab_init(self, dataframe):
    _orig_tab_init(self, dataframe)
    if hasattr(dataframe, '_woodwork_schema') and dataframe._woodwork_schema is not None:
        self._schema = dataframe._woodwork_schema
WoodworkTableAccessor.__init__ = _patched_tab_init

_orig_tab_setattr = WoodworkTableAccessor.__setattr__
def _patched_tab_setattr(self, name, value):
    _orig_tab_setattr(self, name, value)
    if name == '_schema' and value is not None and hasattr(self, '_dataframe'):
        self._dataframe._woodwork_schema = value
WoodworkTableAccessor.__setattr__ = _patched_tab_setattr

_orig_tab_getitem = WoodworkTableAccessor.__getitem__
def _patched_tab_getitem(self, key):
    res = _orig_tab_getitem(self, key)
    if isinstance(key, str) and hasattr(self, '_schema') and self._schema is not None:
        if key in self._schema.columns:
            res._woodwork_schema = self._schema.columns[key]
    return res
WoodworkTableAccessor.__getitem__ = _patched_tab_getitem

# Патч для Series.ww
_orig_col_init = WoodworkColumnAccessor.__init__
def _patched_col_init(self, series):
    _orig_col_init(self, series)
    if hasattr(series, '_woodwork_schema') and series._woodwork_schema is not None:
        self._schema = series._woodwork_schema
WoodworkColumnAccessor.__init__ = _patched_col_init

_orig_col_setattr = WoodworkColumnAccessor.__setattr__
def _patched_col_setattr(self, name, value):
    _orig_col_setattr(self, name, value)
    if name == '_schema' and value is not None:
        series = getattr(self, '_series', None)
        if series is not None:
            series._woodwork_schema = value
WoodworkColumnAccessor.__setattr__ = _patched_col_setattr

np.random.seed(42)

In [2]:
# 1. Таблица customers (клиенты)
n_customers = 50
customers_df = pd.DataFrame({
    'client_id': range(1, n_customers + 1),
    'name': [f'Customer_{i}' for i in range(1, n_customers + 1)],
    'join_date': pd.date_range(start='2023-01-01', periods=n_customers, freq='3D'),
    'region': np.random.choice(['North', 'South', 'East', 'West'], size=n_customers)
})

# 2. Таблица products (товары)
n_products = 20
products_df = pd.DataFrame({
    'product_id': range(1, n_products + 1),
    'category': np.random.choice(['Electronics', 'Clothing', 'Home', 'Books'], size=n_products),
    'price': np.round(np.random.uniform(10, 500, size=n_products), 2)
})

# 3. Таблица orders (заказы)
n_orders = 150
orders_df = pd.DataFrame({
    'order_id': range(1, n_orders + 1),
    'client_id': np.random.choice(customers_df['client_id'], size=n_orders),
    'order_date': pd.date_range(start='2023-06-01', periods=n_orders, freq='18h'),
    'total_amount': np.round(np.random.uniform(20, 1500, size=n_orders), 2)
})

# 4. Таблица order_items (позиции заказов)
n_items = 400
order_items_df = pd.DataFrame({
    'item_id': range(1, n_items + 1),
    'order_id': np.random.choice(orders_df['order_id'], size=n_items),
    'product_id': np.random.choice(products_df['product_id'], size=n_items),
    'quantity': np.random.randint(1, 6, size=n_items)
})

print(f"customers:   {customers_df.shape}")
print(f"orders:      {orders_df.shape}")
print(f"products:    {products_df.shape}")
print(f"order_items: {order_items_df.shape}")

customers:   (50, 4)
orders:      (150, 4)
products:    (20, 3)
order_items: (400, 4)


In [3]:
customers_df.ww.init(name='customers', index='client_id')
orders_df.ww.init(name='orders', index='order_id', time_index='order_date')
products_df.ww.init(name='products', index='product_id')
order_items_df.ww.init(name='order_items', index='item_id')

es = ft.EntitySet(id='retail_store')
es.add_dataframe(dataframe=customers_df)
es.add_dataframe(dataframe=orders_df)
es.add_dataframe(dataframe=products_df)
es.add_dataframe(dataframe=order_items_df)

es.add_relationship('customers', 'client_id', 'orders', 'client_id')
es.add_relationship('orders', 'order_id', 'order_items', 'order_id')
es.add_relationship('products', 'product_id', 'order_items', 'product_id')

print(es)

Entityset: retail_store
  DataFrames:
    customers [Rows: 50, Columns: 4]
    orders [Rows: 150, Columns: 4]
    products [Rows: 20, Columns: 3]
    order_items [Rows: 400, Columns: 4]
  Relationships:
    orders.client_id -> customers.client_id
    order_items.order_id -> orders.order_id
    order_items.product_id -> products.product_id


c:\Users\user\Desktop\lab1_mlops\venv\Lib\site-packages\woodwork\type_sys\utils.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
c:\Users\user\Desktop\lab1_mlops\venv\Lib\site-packages\woodwork\type_sys\utils.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
c:\Users\user\Desktop\lab1_mlops\venv\Lib\site-packages\woodwork\type_sys\utils.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
c:\Users\user\Desktop\lab1_mlops\venv\Lib\site-packages\woodwork\type_sys\utils.py:33: UserWarning: Could not infer format, so each e

In [4]:
feature_matrix, feature_defs = ft.dfs(
    entityset=es,
    target_dataframe_name='customers',
    agg_primitives=['sum', 'mean', 'count', 'max', 'min', 'std'],
    trans_primitives=['day', 'month', 'year'],
    max_depth=2,
    verbose=True
)

Built 46 features
Elapsed: 00:00 | Progress: 100%|██████████


In [5]:
print(f"Размер матрицы признаков: {feature_matrix.shape}")
print(f"Всего сгенерировано признаков: {len(feature_defs)}\n")

for idx, feat in enumerate(feature_defs, 1):
    print(f"{idx:2d}. {feat.get_name()}")

feature_matrix.head()

Размер матрицы признаков: (50, 46)
Всего сгенерировано признаков: 46

 1. region
 2. COUNT(orders)
 3. MAX(orders.total_amount)
 4. MEAN(orders.total_amount)
 5. MIN(orders.total_amount)
 6. STD(orders.total_amount)
 7. SUM(orders.total_amount)
 8. COUNT(order_items)
 9. MAX(order_items.quantity)
10. MEAN(order_items.quantity)
11. MIN(order_items.quantity)
12. STD(order_items.quantity)
13. SUM(order_items.quantity)
14. DAY(join_date)
15. MONTH(join_date)
16. YEAR(join_date)
17. MAX(orders.COUNT(order_items))
18. MAX(orders.MEAN(order_items.quantity))
19. MAX(orders.MIN(order_items.quantity))
20. MAX(orders.STD(order_items.quantity))
21. MAX(orders.SUM(order_items.quantity))
22. MEAN(orders.COUNT(order_items))
23. MEAN(orders.MAX(order_items.quantity))
24. MEAN(orders.MEAN(order_items.quantity))
25. MEAN(orders.MIN(order_items.quantity))
26. MEAN(orders.STD(order_items.quantity))
27. MEAN(orders.SUM(order_items.quantity))
28. MIN(orders.COUNT(order_items))
29. MIN(orders.MAX(order_items

,region,COUNT(orders),MAX(orders.total_amount),MEAN(orders.total_amount),MIN(orders.total_amount),STD(orders.total_amount),SUM(orders.total_amount),COUNT(order_items),MAX(order_items.quantity),MEAN(order_items.quantity),...,STD(orders.SUM(order_items.quantity)),SUM(orders.MAX(order_items.quantity)),SUM(orders.MEAN(order_items.quantity)),SUM(orders.MIN(order_items.quantity)),SUM(orders.STD(order_items.quantity)),MAX(order_items.orders.total_amount),MEAN(order_items.orders.total_amount),MIN(order_items.orders.total_amount),STD(order_items.orders.total_amount),SUM(order_items.orders.total_amount)
client_id,,,,,,,,,,,,,,,,,,,,,
1,East,6,1485.28,670.381667,146.31,553.672269,4022.29,16,5.0,3.562500,...,5.560276,26.0,22.800000,18.0,3.469694,1485.28,621.070000,146.31,544.546388,9937.12
2,West,5,1237.45,664.766000,73.19,376.984168,3323.83,12,5.0,3.583333,...,6.651316,16.0,13.833333,11.0,1.994142,806.40,439.384167,73.19,267.800758,5272.61
3,North,3,1116.34,966.486667,709.09,182.824719,2899.46,12,5.0,3.166667,...,5.557777,14.0,9.100000,4.0,4.270680,1116.34,929.023333,709.09,186.449969,11148.28
4,East,1,763.90,763.900000,763.90,0.000000,763.90,3,4.0,3.000000,...,0.000000,4.0,3.000000,1.0,1.414214,763.90,763.900000,763.90,0.000000,2291.70
5,East,2,803.88,652.735000,501.59,151.145000,1305.47,5,5.0,4.200000,...,3.500000,9.0,8.166667,7.0,0.971405,803.88,622.506000,501.59,148.091251,3112.53
